# Problem 10: Partial Trace and Entropy in a Three-Qubit System

Numerisk verifikation af (b), samt løsning af (c) og (d).

In [4]:
import numpy as np

# Basisorden: |000>,|001>,|010>,|011>,|100>,|101>,|110>,|111>
psi = np.zeros(8)
for state in ["000", "011", "101", "110", "111", "100"]:
    psi[int(state, 2)] = 1
psi = psi / np.sqrt(6)

print("psi =", psi)

# Fuld 8x8 tæthedsmatrix rho = |psi><psi|
rho = np.outer(psi, psi)
np.set_printoptions(precision=4, suppress=True)
print("\nFuld densitetsmatrix rho:")
print(rho)

def von_neumann_entropy(rho):
    eigvals = np.linalg.eigvalsh(rho)
    eigvals = eigvals[eigvals > 1e-12]  # fjern numeriske nuller
    return -np.sum(eigvals * np.log2(eigvals))

psi = [0.4082 0.     0.     0.4082 0.4082 0.4082 0.4082 0.4082]

Fuld densitetsmatrix rho:
[[0.1667 0.     0.     0.1667 0.1667 0.1667 0.1667 0.1667]
 [0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.1667 0.     0.     0.1667 0.1667 0.1667 0.1667 0.1667]
 [0.1667 0.     0.     0.1667 0.1667 0.1667 0.1667 0.1667]
 [0.1667 0.     0.     0.1667 0.1667 0.1667 0.1667 0.1667]
 [0.1667 0.     0.     0.1667 0.1667 0.1667 0.1667 0.1667]
 [0.1667 0.     0.     0.1667 0.1667 0.1667 0.1667 0.1667]]


## (c) Reduceret densitetsmatrix ρ₁ og von Neumann-entropi

Vi sporer qubit 2 og 3 ud ved at reshape ρ til en tensor med indeks (i,j,k,i',j',k') og kontrahere j,k med j',k'.

In [5]:
rho_tensor = rho.reshape(2, 2, 2, 2, 2, 2)

# rho_1[i,i'] = sum_{j,k} rho_tensor[i,j,k,i',j,k]
rho1 = np.einsum('ijkljk->il', rho_tensor)

print("Reduceret densitetsmatrix rho_1 (qubit 1):")
print(rho1)
print("\nTr(rho_1) =", np.trace(rho1).real)
print("Egenvaerdier:", np.linalg.eigvalsh(rho1))

S1 = von_neumann_entropy(rho1)
print(f"\nVon Neumann entropi S(rho_1) = {S1:.6f} bit")

Reduceret densitetsmatrix rho_1 (qubit 1):
[[0.3333 0.3333]
 [0.3333 0.6667]]

Tr(rho_1) = 1.0000000000000002
Egenvaerdier: [0.1273 0.8727]

Von Neumann entropi S(rho_1) = 0.550048 bit


## (d) Reduceret densitetsmatrix ρ₁₂ og von Neumann-entropi

Nu spores kun qubit 3 ud.

In [6]:
# rho_12[(i,j),(i',j')] = sum_k rho_tensor[i,j,k,i',j',k]
rho12_tensor = np.einsum('ijkabk->ijab', rho_tensor)
rho12 = rho12_tensor.reshape(4, 4)

print("Reduceret densitetsmatrix rho_12 (qubit 1,2):")
print(rho12)
print("\nTr(rho_12) =", np.trace(rho12).real)
print("Egenvaerdier:", np.linalg.eigvalsh(rho12))

S12 = von_neumann_entropy(rho12)
print(f"\nVon Neumann entropi S(rho_12) = {S12:.6f} bit")

Reduceret densitetsmatrix rho_12 (qubit 1,2):
[[0.1667 0.     0.1667 0.1667]
 [0.     0.1667 0.1667 0.1667]
 [0.1667 0.1667 0.3333 0.3333]
 [0.1667 0.1667 0.3333 0.3333]]

Tr(rho_12) = 1.0000000000000002
Egenvaerdier: [-0.      0.      0.1667  0.8333]

Von Neumann entropi S(rho_12) = 0.650022 bit


## Sammenligning

S(ρ₁) ≈ 0.5500 bit, S(ρ₁₂) ≈ 0.6500 bit. Begge er strengt mellem 0 og 1 bit ⇒ tilstanden er sammenfiltret for begge opdelinger.

Da den globale tilstand |ψ⟩ er ren, gælder S(ρ_A) = S(ρ_Ā) for komplementære undersystemer. Derfor er S(ρ₁₂) = S(ρ₃), og de to tal ovenfor svarer altså til to *forskellige* bipartitioner (1 mod 23, og 12 mod 3) — de er ikke direkte sammenlignelige som "samme snit", men begge illustrerer sammenfiltringen i systemet.